# LLM Observability with Langfuse
## Tracing AI Agent Decisions — A Beginner's Guide

Welcome! In this notebook you will learn:

- **What** LLM observability is and **why** it matters
- How to run **Langfuse** locally with Docker
- How to trace a simple LLM call
- How to build a **ReAct agent** that logs every decision it makes
- How to read the trace tree in the **Langfuse UI**

---

**Tech stack used in this notebook**

| Component | What it does |
|-----------|-------------|
| `claude-haiku-4-5` | Fast, cheap Anthropic model — perfect for demos |
| `anthropic` SDK | Python client for calling Claude |
| `langfuse` SDK | Python client for sending traces to Langfuse |
| Langfuse (Docker) | Local UI + storage for all your traces |


---

## 1. What is LLM Observability?

Imagine your AI agent gives a user the wrong answer. How do you find out what went wrong?

- Did the model misunderstand the question?
- Did it call the wrong tool?
- Did a tool return bad data?
- Did it fail on step 3 of 7 but you only see the final wrong answer?

**Observability** is the practice of recording *everything* an AI system does at runtime so you can answer these questions after the fact.

### The flight data recorder analogy

Think of it like a **black box recorder** on an aircraft.  
The plane records every sensor, every pilot action, every system state — not because something is wrong, but so that *if* something goes wrong you have a complete record to investigate.

For LLM apps the recorder captures:

| What is recorded | Why it matters |
|------------------|----------------|
| Every LLM call (input + output) | Debug wrong or hallucinated answers |
| Token counts per call | Track and reduce API cost |
| Latency per step | Find performance bottlenecks |
| Tool calls + results | Trace exactly what the agent decided to do |
| Full multi-step sessions | Understand end-to-end user journeys |


---

## 2. Why Does Observability Matter for LLM Agents?

### Without observability (flying blind)

```
User  : "What is 25% of 340?"
Agent : "The answer is 255"   ← WRONG. But why?
Dev   :  ¯\_(ツ)_/¯
```

### With observability (full visibility)

You open Langfuse and see the exact trace:

```
ReAct Agent
├── Step 1 — Reasoning
│   ├── LLM Call  →  "I'll calculate 25% of 340"
│   └── Tool: calculator(multiply, 340, 0.25)  →  85
├── Step 2 — Reasoning
│   ├── LLM Call  →  "Now subtract: 340 - 85"   ← BUG: misunderstood 25%
│   └── Tool: calculator(subtract, 340, 85)  →  255
└── Final answer: 255
```

Now you know **exactly** what happened: the agent subtracted instead of just returning the multiplication result.
You can fix your system prompt immediately.

### Real-world benefits

- **Cost control** — See which queries burn the most tokens
- **Quality monitoring** — Catch regressions when you change a prompt
- **Debugging** — Replay the exact input/output that caused a failure
- **Performance** — Spot slow tool calls or bottleneck steps
- **Audit trail** — Record of every AI decision for compliance


---

## 3. What is Langfuse?

**Langfuse** is an open-source LLM observability platform. Think of it as a specialized logging dashboard built specifically for AI applications.

### Key concepts in Langfuse

| Concept | What it is | Analogy |
|---------|-----------|--------|
| **Trace** | One end-to-end request (e.g. one agent run) | A single flight log |
| **Span** | A step inside a trace (e.g. "call tool X") | One flight manoeuvre |
| **Generation** | An LLM call with input, output, and token counts | Pilot radio call |
| **Session** | A group of traces (e.g. one user conversation) | One trip |
| **Score** | A quality rating attached to a trace | Post-flight review |

### How data flows

```
Your Python code
      │  langfuse.trace() / span() / generation()
      ▼
Langfuse SDK  ──(HTTP)──►  Langfuse Server (Docker :3000)
                                    │
                                    ▼
                             PostgreSQL DB
                                    │
                                    ▼
                            Langfuse Web UI
                         (you browse traces here)
```


---

## Setup

### Step 1 — Start Langfuse with Docker

Open a terminal in the same folder as this notebook and run:

```bash
docker compose up -d
```

This starts two containers:
- `langfuse` — the web app (port 3000)
- `postgres` — the database

Wait about **30–60 seconds** for the app to finish compiling on first start.  
Then open **http://localhost:3000** in your browser.

### Step 2 — Create an account and get API keys

1. Click **Sign Up** on the Langfuse login page
2. Fill in any email + password (this is local-only, no internet required)
3. Click **Create new project** and give it a name, e.g. `llm-observability-demo`
4. In the left sidebar go to **Settings → API Keys**
5. Click **Create new API key**
6. Copy both the **Public Key** (`pk-lf-...`) and **Secret Key** (`sk-lf-...`)

### Step 3 — Create your `.env` file

Copy the example file and fill in your keys:

```bash
cp .env.example .env
# then edit .env with your keys
```

Your `.env` should look like:

```
ANTHROPIC_API_KEY=sk-ant-...
LANGFUSE_HOST=http://localhost:3000
LANGFUSE_PUBLIC_KEY=pk-lf-...
LANGFUSE_SECRET_KEY=sk-lf-...
```


In [1]:
# Install required packages
# Run this once — you can skip it on subsequent runs
%pip install anthropic langfuse python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
from dotenv import load_dotenv
import anthropic
from langfuse import Langfuse

# Load credentials from .env
load_dotenv(".env")

# ── Langfuse client ──────────────────────────────────────────────────────────
# Connects to your local Docker instance.
# All traces will appear at http://localhost:3000
langfuse = Langfuse(
    public_key=os.getenv('LANGFUSE_PUBLIC_KEY'),
    secret_key=os.getenv('LANGFUSE_SECRET_KEY'),
    host=os.getenv('LANGFUSE_HOST', 'http://localhost:3000'),
)

# ── Anthropic client ─────────────────────────────────────────────────────────
claude = anthropic.Anthropic(
    api_key=os.getenv('ANTHROPIC_API_KEY')
)

MODEL = 'claude-haiku-4-5-20251001'

print('Clients ready.')
print(f'  Langfuse : {os.getenv("LANGFUSE_HOST", "http://localhost:3000")}')
print(f'  Model    : {MODEL}')

Clients ready.
  Langfuse : http://localhost:3000
  Model    : claude-haiku-4-5-20251001


In [3]:
# Test the Langfuse connection
# If this fails, check that Docker is running and your API keys are correct.
try:
    langfuse.auth_check()
    print('Langfuse connection: OK')
    print(f'  Open the UI at: {os.getenv("LANGFUSE_HOST", "http://localhost:3000")}')
except Exception as e:
    print(f'Connection failed: {e}')
    print()
    print('Troubleshooting:')
    print('  1. docker compose up -d          (start containers)')
    print('  2. Wait 30-60 seconds')
    print('  3. Open http://localhost:3000    (create account + project)')
    print('  4. Copy API keys into .env')
    print('  5. Re-run this cell')

Langfuse connection: OK
  Open the UI at: http://localhost:3000


---

## Part 1 — Tracing a Simple LLM Call

Before building the full agent, let's trace a **single LLM call**.  
This shows the most basic observability pattern:

```
Trace  ──►  Generation
```

- **Trace** = the overall request (one entry in the Langfuse UI)
- **Generation** = the LLM call inside it (records model, tokens, input, output)


In [4]:
def traced_llm_call(user_message: str) -> str:
    """
    Makes a single Claude call and records it in Langfuse.
    After running this, refresh http://localhost:3000/traces to see it.
    """

    # 1. Create a trace — the top-level container for this request
    trace = langfuse.trace(
        name='simple-llm-call',
        input={'message': user_message},
        tags=['simple', 'beginner-demo'],
    )

    # 2. Create a generation — records the LLM call specifically
    #    (model name, input messages, output, token counts)
    generation = trace.generation(
        name='claude-response',
        model=MODEL,
        input=[{'role': 'user', 'content': user_message}],
    )

    # 3. Make the actual API call
    response = claude.messages.create(
        model=MODEL,
        max_tokens=512,
        messages=[{'role': 'user', 'content': user_message}],
    )

    answer = response.content[0].text

    # 4. End the generation — attach the output and token usage
    generation.end(
        output=answer,
        usage={
            'input':  response.usage.input_tokens,
            'output': response.usage.output_tokens,
        },
    )

    # 5. Save the final answer on the trace itself
    trace.update(output={'answer': answer})

    # flush() makes sure all buffered events are sent to Langfuse now
    langfuse.flush()

    print(f'Q: {user_message}')
    print(f'A: {answer}')
    print(f'\nTokens used — input: {response.usage.input_tokens}, output: {response.usage.output_tokens}')
    print(f'View trace : {os.getenv("LANGFUSE_HOST", "http://localhost:3000")}/traces')
    return answer


# Run it!
traced_llm_call('What is the capital of Japan, and what is it famous for?')

Q: What is the capital of Japan, and what is it famous for?
A: # Tokyo

**Capital:** Tokyo is the capital of Japan.

**What it's famous for:**

- **Modern skyline** – Iconic skyscrapers, neon lights, and futuristic architecture
- **Cultural blend** – Mix of ancient temples and shrines with ultra-modern technology
- **Food scene** – World-class restaurants, street food, and Michelin-starred dining
- **Shopping districts** – Shibuya, Shinjuku, and Ginza are globally renowned
- **Entertainment** – Anime, gaming, fashion, and pop culture hub
- **Transport** – Efficient and extensive train and subway system
- **Landmarks** – Senso-ji Temple, Tokyo Tower, Imperial Palace, and Tsukiji Market
- **Population** – One of the world's largest metropolitan areas

Tokyo perfectly represents Japan's unique combination of traditional culture and cutting-edge modernity.

Tokens used — input: 21, output: 202
View trace : http://localhost:3000/traces


"# Tokyo\n\n**Capital:** Tokyo is the capital of Japan.\n\n**What it's famous for:**\n\n- **Modern skyline** – Iconic skyscrapers, neon lights, and futuristic architecture\n- **Cultural blend** – Mix of ancient temples and shrines with ultra-modern technology\n- **Food scene** – World-class restaurants, street food, and Michelin-starred dining\n- **Shopping districts** – Shibuya, Shinjuku, and Ginza are globally renowned\n- **Entertainment** – Anime, gaming, fashion, and pop culture hub\n- **Transport** – Efficient and extensive train and subway system\n- **Landmarks** – Senso-ji Temple, Tokyo Tower, Imperial Palace, and Tsukiji Market\n- **Population** – One of the world's largest metropolitan areas\n\nTokyo perfectly represents Japan's unique combination of traditional culture and cutting-edge modernity."

**Go to http://localhost:3000 → Traces** and you should see a new entry called `simple-llm-call`.  
Click it to see the generation with input, output, and token counts.

---

## Part 2 — Tracing a ReAct Agent

Now for the interesting part. We will build an agent that can use tools and trace **every decision it makes**.

### What is the ReAct pattern?

**ReAct** (Reason + Act) is how modern LLM agents work:

```
User query
    │
    ▼
┌─────────────────────────────────────────────┐
│  LOOP (repeats until final answer)          │
│                                             │
│  1. REASON  → LLM reads the conversation   │
│               and decides what to do next   │
│                                             │
│  2. ACT     → If LLM picks a tool:         │
│               we run it and get the result  │
│                                             │
│  3. OBSERVE → Tool result goes back to LLM  │
│                                             │
│  (stop when LLM returns a final answer)     │
└─────────────────────────────────────────────┘
    │
    ▼
Final answer
```

### What the trace tree looks like in Langfuse

```
Trace: ReAct Agent
├── Span:  Step 1 — Reasoning
│   ├── Generation: LLM Call   (input/output/tokens logged here)
│   └── Span:       Tool: get_weather   (tool input + result logged here)
├── Span:  Step 2 — Reasoning
│   ├── Generation: LLM Call
│   └── Span:       Tool: unit_converter
└── Span:  Step 3 — Reasoning
    └── Generation: LLM Call   (stop_reason = end_turn → final answer)
```

Every box above is one clickable row in the Langfuse UI.


In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# Tool definitions — these are sent to Claude so it knows what tools exist
# and what arguments each one expects.
# ─────────────────────────────────────────────────────────────────────────────

TOOLS = [
    {
        'name': 'calculator',
        'description': (
            'Performs basic arithmetic: add, subtract, multiply, divide. '
            'Use this whenever the user asks for a calculation.'
        ),
        'input_schema': {
            'type': 'object',
            'properties': {
                'operation': {
                    'type': 'string',
                    'enum': ['add', 'subtract', 'multiply', 'divide'],
                    'description': 'The arithmetic operation to perform',
                },
                'a': {'type': 'number', 'description': 'First number'},
                'b': {'type': 'number', 'description': 'Second number'},
            },
            'required': ['operation', 'a', 'b'],
        },
    },
    {
        'name': 'get_weather',
        'description': (
            'Returns the current weather for a city. '
            'Provides temperature (Celsius), condition, and humidity. '
            'Note: uses simulated data for this demo.'
        ),
        'input_schema': {
            'type': 'object',
            'properties': {
                'city': {
                    'type': 'string',
                    'description': 'City name, e.g. "London" or "Tokyo"',
                },
            },
            'required': ['city'],
        },
    },
    {
        'name': 'unit_converter',
        'description': (
            'Converts a value between units. '
            'Supported pairs: celsius↔fahrenheit, km↔miles, kg↔lbs.'
        ),
        'input_schema': {
            'type': 'object',
            'properties': {
                'value':     {'type': 'number', 'description': 'Value to convert'},
                'from_unit': {'type': 'string', 'description': 'Source unit (celsius, fahrenheit, km, miles, kg, lbs)'},
                'to_unit':   {'type': 'string', 'description': 'Target unit'},
            },
            'required': ['value', 'from_unit', 'to_unit'],
        },
    },
]

print(f'Defined {len(TOOLS)} tools: {[t["name"] for t in TOOLS]}')

Defined 3 tools: ['calculator', 'get_weather', 'unit_converter']


In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# Tool implementations (dummy / simulated data — no real APIs needed)
# ─────────────────────────────────────────────────────────────────────────────

def calculator(operation: str, a: float, b: float) -> dict:
    ops = {
        'add':      lambda: a + b,
        'subtract': lambda: a - b,
        'multiply': lambda: a * b,
        'divide':   lambda: a / b if b != 0 else None,
    }
    if operation not in ops:
        return {'error': f'Unknown operation: {operation}'}
    result = ops[operation]()
    if result is None:
        return {'error': 'Cannot divide by zero'}
    return {'result': round(result, 6), 'expression': f'{a} {operation} {b} = {round(result, 6)}'}


def get_weather(city: str) -> dict:
    # Simulated weather data — swap with a real API in production
    data = {
        'london':   {'temp_c': 14, 'condition': 'Cloudy',        'humidity': 78},
        'new york': {'temp_c': 21, 'condition': 'Sunny',         'humidity': 55},
        'tokyo':    {'temp_c': 27, 'condition': 'Partly Cloudy', 'humidity': 80},
        'paris':    {'temp_c': 17, 'condition': 'Rainy',         'humidity': 88},
        'sydney':   {'temp_c': 23, 'condition': 'Clear',         'humidity': 60},
        'dubai':    {'temp_c': 38, 'condition': 'Hot & Sunny',   'humidity': 45},
    }
    d = data.get(city.lower().strip(), {'temp_c': 20, 'condition': 'Clear', 'humidity': 65})
    return {
        'city':                  city,
        'temperature_celsius':   d['temp_c'],
        'condition':             d['condition'],
        'humidity_percent':      d['humidity'],
        'note':                  '(simulated data)',
    }


def unit_converter(value: float, from_unit: str, to_unit: str) -> dict:
    from_unit = from_unit.lower().strip()
    to_unit   = to_unit.lower().strip()
    table = {
        ('celsius',    'fahrenheit'): lambda x: round(x * 9/5 + 32, 2),
        ('fahrenheit', 'celsius'):    lambda x: round((x - 32) * 5/9, 2),
        ('km',         'miles'):      lambda x: round(x * 0.621371, 4),
        ('miles',      'km'):         lambda x: round(x * 1.60934, 4),
        ('kg',         'lbs'):        lambda x: round(x * 2.20462, 4),
        ('lbs',        'kg'):         lambda x: round(x * 0.453592, 4),
    }
    key = (from_unit, to_unit)
    if key not in table:
        supported = [f'{a} -> {b}' for a, b in table]
        return {'error': f'No conversion for {from_unit} -> {to_unit}', 'supported': supported}
    result = table[key](value)
    return {'original': f'{value} {from_unit}', 'result': result, 'unit': to_unit}


def execute_tool(name: str, args: dict) -> dict:
    """Dispatches a tool call to the right Python function."""
    registry = {
        'calculator':    calculator,
        'get_weather':   get_weather,
        'unit_converter': unit_converter,
    }
    if name not in registry:
        return {'error': f'Unknown tool: {name}'}
    try:
        return registry[name](**args)
    except Exception as exc:
        return {'error': str(exc)}


# Quick smoke test
print('calculator:     ', calculator('multiply', 340, 0.25))
print('get_weather:    ', get_weather('tokyo'))
print('unit_converter: ', unit_converter(14, 'celsius', 'fahrenheit'))

calculator:      {'result': 85.0, 'expression': '340 multiply 0.25 = 85.0'}
get_weather:     {'city': 'tokyo', 'temperature_celsius': 27, 'condition': 'Partly Cloudy', 'humidity_percent': 80, 'note': '(simulated data)'}
unit_converter:  {'original': '14 celsius', 'result': 57.2, 'unit': 'fahrenheit'}


In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# ReAct Agent with full Langfuse tracing
# ─────────────────────────────────────────────────────────────────────────────

def run_agent(user_query: str, session_id: str = None) -> str:
    """
    Runs a ReAct agent and traces every decision in Langfuse.

    Trace structure produced:

        Trace: ReAct Agent
        ├── Span:       Step 1 — Reasoning
        │   ├── Generation: LLM Call          ← tokens, input, output
        │   └── Span:       Tool: <name>      ← args + result
        ├── Span:       Step 2 — Reasoning
        │   ├── Generation: LLM Call
        │   └── Span:       Tool: <name>
        └── ...
    """
    MAX_STEPS = 8

    # ── 1. Create the top-level trace ────────────────────────────────────────
    # Everything that happens during this agent run is nested under this trace.
    trace = langfuse.trace(
        name='ReAct Agent',
        input={'query': user_query},
        session_id=session_id,
        tags=['react-agent', 'demo'],
        metadata={
            'model':           MODEL,
            'max_steps':       MAX_STEPS,
            'available_tools': [t['name'] for t in TOOLS],
        },
    )

    messages = [{'role': 'user', 'content': user_query}]

    host = os.getenv('LANGFUSE_HOST', 'http://localhost:3000')
    print('=' * 60)
    print(f'QUERY : {user_query}')
    print(f'TRACE : {host}/traces/{trace.id}')
    print('=' * 60)

    for step in range(1, MAX_STEPS + 1):
        print(f'\n--- Step {step} ---')

        # ── 2. Span for this reasoning step ──────────────────────────────────
        # A span marks the start and end of a logical unit of work.
        step_span = trace.span(
            name=f'Step {step} \u2014 Reasoning',
            input={'messages_in_context': len(messages), 'step': step},
        )

        # ── 3. Generation: log the LLM call ──────────────────────────────────
        # A generation is a special span for LLM calls.
        # Langfuse uses it to calculate cost automatically.
        generation = step_span.generation(
            name='LLM Call',
            model=MODEL,
            input=messages,
            metadata={'step': step},
        )

        # Call Claude
        response = claude.messages.create(
            model=MODEL,
            max_tokens=1024,
            tools=TOOLS,
            messages=messages,
        )

        # Serialize content blocks for Langfuse (avoid Pydantic model objects)
        output_log = []
        for block in response.content:
            if block.type == 'text':
                output_log.append({'type': 'text', 'text': block.text})
            elif block.type == 'tool_use':
                output_log.append({'type': 'tool_use', 'name': block.name, 'input': block.input})

        # End the generation — attach output and token usage
        generation.end(
            output=output_log,
            usage={
                'input':  response.usage.input_tokens,
                'output': response.usage.output_tokens,
            },
        )

        print(f'  stop_reason : {response.stop_reason}')
        print(f'  tokens      : in={response.usage.input_tokens} out={response.usage.output_tokens}')

        # ── 4. Agent is done: return the final answer ─────────────────────────
        if response.stop_reason == 'end_turn':
            final_text = next(
                (b.text for b in response.content if b.type == 'text'),
                '(no text in response)'
            )
            print(f'  FINAL ANSWER: {final_text}')

            step_span.end(output={'decision': 'final_answer', 'answer': final_text})
            trace.update(
                output={'answer': final_text, 'steps_taken': step},
                metadata={'completed': True},
            )
            langfuse.flush()
            return final_text

        # ── 5. Agent wants to use a tool ──────────────────────────────────────
        if response.stop_reason == 'tool_use':
            tool_blocks = [b for b in response.content if b.type == 'tool_use']

            # Append Claude's response (including tool_use blocks) to the conversation
            messages.append({'role': 'assistant', 'content': response.content})

            tool_results = []
            for tb in tool_blocks:
                print(f'  TOOL CALL : {tb.name}({tb.input})')

                # ── 6. Span: log tool execution ───────────────────────────────
                tool_span = step_span.span(
                    name=f'Tool: {tb.name}',
                    input={'tool': tb.name, 'args': tb.input},
                    metadata={'tool_name': tb.name},
                )

                result = execute_tool(tb.name, tb.input)
                print(f'  TOOL RESULT: {result}')

                tool_span.end(output={'result': result})

                tool_results.append({
                    'type':        'tool_result',
                    'tool_use_id': tb.id,
                    'content':     json.dumps(result),
                })

            step_span.end(output={
                'decision':    'tool_use',
                'tools_called': [tb.name for tb in tool_blocks],
            })

            # Append tool results so Claude sees them on the next iteration
            messages.append({'role': 'user', 'content': tool_results})

    # Ran out of steps
    trace.update(
        output={'error': 'max_steps_reached'},
        metadata={'completed': False, 'steps_taken': MAX_STEPS},
    )
    langfuse.flush()
    return 'Agent stopped: maximum steps reached'


print('Agent function defined and ready.')

Agent function defined and ready.


---

## Run the Agent — Example Queries

Each cell below runs the agent with a different query.  
After each run, click the trace URL printed in the output to inspect it in Langfuse.


In [8]:
# Example 1 — Pure calculation
# Expected tool calls: calculator (likely twice)
# Teaches you: how multi-step arithmetic is traced

result = run_agent(
    'What is 15% of 840? Then multiply that result by 3.',
    session_id='demo-session-math'
)
print(f'\nFinal: {result}')

QUERY : What is 15% of 840? Then multiply that result by 3.
TRACE : http://localhost:3000/traces/5d80b4dc-09df-4aa9-974d-44fb7eee6c2c

--- Step 1 ---
  stop_reason : tool_use
  tokens      : in=900 out=88
  TOOL CALL : calculator({'operation': 'multiply', 'a': 840, 'b': 0.15})
  TOOL RESULT: {'result': 126.0, 'expression': '840 multiply 0.15 = 126.0'}

--- Step 2 ---
  stop_reason : tool_use
  tokens      : in=1026 out=96
  TOOL CALL : calculator({'operation': 'multiply', 'a': 126, 'b': 3})
  TOOL RESULT: {'result': 378, 'expression': '126 multiply 3 = 378'}

--- Step 3 ---
  stop_reason : end_turn
  tokens      : in=1154 out=47
  FINAL ANSWER: Perfect! Here are the results:

1. **15% of 840** = **126**
2. **126 × 3** = **378**

So the final answer is **378**.

Final: Perfect! Here are the results:

1. **15% of 840** = **126**
2. **126 × 3** = **378**

So the final answer is **378**.


In [9]:
# Example 2 — Weather + unit conversion
# Expected tool calls: get_weather → unit_converter
# Teaches you: how multi-tool chains appear in the trace tree

result = run_agent(
    'What is the weather in Tokyo right now? Also convert the temperature to Fahrenheit for me.',
    session_id='demo-session-weather'
)
print(f'\nFinal: {result}')

QUERY : What is the weather in Tokyo right now? Also convert the temperature to Fahrenheit for me.
TRACE : http://localhost:3000/traces/cb96371e-488b-4986-8a9a-3c957a051fb3

--- Step 1 ---
  stop_reason : tool_use
  tokens      : in=903 out=73
  TOOL CALL : get_weather({'city': 'Tokyo'})
  TOOL RESULT: {'city': 'Tokyo', 'temperature_celsius': 27, 'condition': 'Partly Cloudy', 'humidity_percent': 80, 'note': '(simulated data)'}

--- Step 2 ---
  stop_reason : tool_use
  tokens      : in=1028 out=104
  TOOL CALL : unit_converter({'value': 27, 'from_unit': 'celsius', 'to_unit': 'fahrenheit'})
  TOOL RESULT: {'original': '27 celsius', 'result': 80.6, 'unit': 'fahrenheit'}

--- Step 3 ---
  stop_reason : end_turn
  tokens      : in=1167 out=60
  FINAL ANSWER: Here's the current weather in Tokyo:

- **Temperature**: 27°C (80.6°F)
- **Condition**: Partly Cloudy
- **Humidity**: 80%

It's a warm and humid day in Tokyo right now!

Final: Here's the current weather in Tokyo:

- **Temperature**: 2

In [10]:
# Example 3 — Multi-step, multi-tool
# Expected tool calls: get_weather, unit_converter, calculator
# Teaches you: complex agent reasoning traced end-to-end

result = run_agent(
    'Check the weather in London. '
    'Convert the temperature to Fahrenheit. '
    'Then tell me: if I burn 8 calories per minute jogging, '
    'how many calories would I burn in 45 minutes?',
    session_id='demo-session-complex'
)
print(f'\nFinal: {result}')

QUERY : Check the weather in London. Convert the temperature to Fahrenheit. Then tell me: if I burn 8 calories per minute jogging, how many calories would I burn in 45 minutes?
TRACE : http://localhost:3000/traces/d5ed7c2a-a31c-45f5-97c4-36e5f47bca2c

--- Step 1 ---
  stop_reason : tool_use
  tokens      : in=925 out=73
  TOOL CALL : get_weather({'city': 'London'})
  TOOL RESULT: {'city': 'London', 'temperature_celsius': 14, 'condition': 'Cloudy', 'humidity_percent': 78, 'note': '(simulated data)'}

--- Step 2 ---
  stop_reason : tool_use
  tokens      : in=1048 out=182
  TOOL CALL : unit_converter({'value': 14, 'from_unit': 'celsius', 'to_unit': 'fahrenheit'})
  TOOL RESULT: {'original': '14 celsius', 'result': 57.2, 'unit': 'fahrenheit'}
  TOOL CALL : calculator({'operation': 'multiply', 'a': 8, 'b': 45})
  TOOL RESULT: {'result': 360, 'expression': '8 multiply 45 = 360'}

--- Step 3 ---
  stop_reason : end_turn
  tokens      : in=1334 out=98
  FINAL ANSWER: Perfect! Here are your an

---

## Viewing Traces in the Langfuse UI

Open **http://localhost:3000** and follow these steps.

### 1 — Find your traces

Click **Traces** in the left sidebar.  
You should see one row per `run_agent()` call, plus the simple call from Part 1.

The list view shows:
- **Name** — the trace name you passed (`ReAct Agent`)
- **Timestamp** — when it ran
- **Latency** — total time for the whole agent run
- **Cost** — automatically calculated from token counts + model pricing
- **Tags** — the tags you passed (`react-agent`, `demo`)

### 2 — Open a trace

Click any row to open the detail view.  
You will see the **trace tree** — a nested list of everything that happened:

```
ReAct Agent                           [total latency]
├── Step 1 — Reasoning                [span latency]
│   ├── LLM Call                      [tokens: in=120 out=45]
│   └── Tool: get_weather             [2 ms]
├── Step 2 — Reasoning
│   ├── LLM Call                      [tokens: in=200 out=60]
│   └── Tool: unit_converter          [1 ms]
└── Step 3 — Reasoning
    └── LLM Call                      [tokens: in=280 out=80]
```

### 3 — What each field means

| Field | Where to look | Description |
|-------|--------------|-------------|
| **Input** | Click any row → left panel | Exactly what was sent in |
| **Output** | Click any row → right panel | Exactly what came back |
| **Tokens** | Generation rows | Input + output token counts |
| **Cost** | Generation rows | Calculated from token counts |
| **Latency** | Any row | Wall-clock time for that step |
| **Metadata** | Bottom of detail panel | Custom data you attached |
| **Tags** | Top of trace | Labels for filtering |

### 4 — Sessions view

Click **Sessions** in the sidebar.  
The three example runs above each used a different `session_id`, so you can see them grouped.

### 5 — Filter and search

In the Traces list, use the filter bar to narrow down by:
- **Tag** → type `react-agent` to see only agent runs
- **Date range** → traces from the last hour
- **Name** → `ReAct Agent` or `simple-llm-call`


---

## Summary

Here is everything you learned in this notebook:

| Concept | What you did |
|---------|-------------|
| **Observability** | Understand why recording LLM decisions matters |
| **Langfuse setup** | Run a full observability stack with 2 Docker containers |
| **Traces** | Create a top-level container for each agent run |
| **Spans** | Wrap each reasoning step in a timed, inspectable unit |
| **Generations** | Log every LLM call with model, tokens, input, and output |
| **Tool tracing** | Record tool arguments and results inside spans |
| **Sessions** | Group related traces with a session_id |
| **UI navigation** | Browse traces, inspect inputs/outputs, filter by tag |

### Next steps

- **Add scores** — Call `trace.score(name='quality', value=0.9)` to rate traces
- **Add user IDs** — Pass `user_id='user-123'` to `langfuse.trace()` to track per-user behaviour
- **Use the decorator** — `from langfuse.decorators import observe` for automatic tracing with less boilerplate
- **Prompt management** — Store and version your prompts in Langfuse and pull them at runtime
- **Real tools** — Replace the dummy `get_weather` with a real weather API call
- **Production** — Replace `NEXTAUTH_SECRET` with a strong random value and run behind HTTPS

### Useful links

- Langfuse docs: https://langfuse.com/docs
- Anthropic tool use: https://docs.anthropic.com/en/docs/tool-use
- Langfuse Python SDK: https://langfuse.com/docs/sdk/python
